## GitHub에서 이 노트북을 열 때 (Colab / 원격)

1. 아래 **다음 셀**에서 `REPO_URL` 을 **본인 저장소 HTTPS 주소**로 바꾼 뒤 실행합니다. (한 세션에 한 번)
2. 그다음 **import 셀**을 위에서부터 순서대로 실행합니다.
3. 이미 로컬에서 이 레포를 연 상태면 **clone 셀은 실행하지 않아도** 됩니다 (import 셀만 실행).

저장소를 올릴 때는 루트에 `pyproject.toml`, `src/mindscopex_analysis/` 전체가 포함되어야 합니다. 자세한 협업 맥락은 저장소 루트의 **`CLAUDE.md`** 를 참고하세요.

In [ ]:
# @title 0) GitHub에서 저장소 클론 (Colab·원격 — 로컬에서 이미 열었으면 이 셀 생략)
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ▼▼▼ 푸시한 뒤 본인 저장소 URL로 수정 ▼▼▼
REPO_URL = "https://github.com/YOUR_ORG/YOUR_REPO.git"
# Colab 기본. 바꾸려면: os.environ["COLAB_REPO_DIR"] = "/content/my_colab"
WORKDIR = Path(os.environ.get("COLAB_REPO_DIR", "/content/colab"))

MARK = WORKDIR / "src" / "mindscopex_analysis" / "__init__.py"


def _run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


if MARK.is_file():
    print(f"이미 클론됨 → git pull 만 실행: {WORKDIR}")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=False)
else:
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    _run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)])

os.environ["MINDSCOPEX_ROOT"] = str(WORKDIR.resolve())
os.chdir(WORKDIR)
print("cwd =", os.getcwd())
print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"])

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "torch",
        "transformers",
        "accelerate",
        "plotly",
        "pandas",
        "pyyaml",
    ]
)
print("pip install -e . 완료")

# Reasoning vs Non-Reasoning: 뉴런 활성화 비교

**목표**: 같은 패밀리·비슷한 규모의 모델들에 **서로 다른 프롬프트 그룹**(예: 직관 vs 논리)을 넣었을 때
레이어·뉴런 활성화 패턴을 비교합니다.

**조절 방법**
1. 아래 **실험 설정** 셀에서 `ACTIVE_PRESET`만 바꿔 모델군을 전환 (`notebook_presets.py`에 프리셋 추가 가능).
2. 선택적으로 `configs/notebook_neuron_compare.yaml`을 열어 `preset`, `runtime`, `analysis`를 덮어씁니다 (병합).
3. 분석 레이어는 `analysis.target_layer`를 `mid` / `first` / `last` 또는 정수 레이어 인덱스로 지정.

> VRAM이 작으면 `qwen_single_instruct` 또는 0.5B 쌍을 권장합니다.

### Colab / 원격 커널에서 `No module named 'mindscopex_analysis'` 일 때

원격(예: Colab)에서는 **현재 작업 폴더가 로컬과 다릅니다.** 아래 중 하나를 쓰면 됩니다.

1. **프로젝트 루트로 이동** — 저장소를 clone·업로드한 뒤 그 디렉터리에서 노트북을 열거나, 첫 셀 실행 전에  
   `%cd /content/당신의/colab` (경로는 환경에 맞게) 로 루트로 이동합니다.  
   첫 코드 셀은 상위 디렉터리를 따라가며 `src/mindscopex_analysis` 을 찾습니다.

2. **환경 변수** — 루트를 고정하려면 첫 셀 **맨 위**에  
   `import os` 후 `os.environ["MINDSCOPEX_ROOT"] = "/content/당신의/colab"`  
   (그 아래에 `src/mindscopex_analysis/__init__.py` 가 있어야 합니다.)

3. **editable 설치** — 루트에 `pyproject.toml` 이 있다면 터미널에서  
   `pip install -e .`  
   를 한 번 실행하면 `mindscopex_analysis` 이 패키지로 등록되어 경로 조정 없이 import 됩니다.

4. **설정 YAML** — `configs/notebook_neuron_compare.yaml` 을 쓰려면 위 1–3으로 **전체 저장소**가 보이는 상태여야 합니다.

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
from copy import deepcopy
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Colab/원격: cwd 가 프로젝트 루트가 아닐 수 있음 → src/mindscopex_analysis 까지 올라가며 탐색 ---
_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
        raise FileNotFoundError(
            f"MINDSCOPEX_ROOT={env!r} 아래에 {_REPO_MARK.as_posix()} 가 없습니다."
        )
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError(
        "src/mindscopex_analysis 을 찾지 못했습니다. Colab에서는 clone 후 %cd 로 프로젝트 루트로 이동하거나 "
        "os.environ['MINDSCOPEX_ROOT']='/path/to/colab' 을 설정하세요."
    )


_SRC = _find_repo_root() / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from mindscopex_analysis.neurons import (
    per_neuron_stats,
    top_k_neurons,
    differential_neurons,
    cosine_similarity_matrix,
)
from mindscopex_analysis.sae.trainer import SAETrainer, SAETrainConfig
from mindscopex_analysis.notebook_utils import (
    project_root_from_notebook,
    deep_merge,
    dtype_from_str,
    resolve_target_layer,
)
from mindscopex_analysis.notebook_presets import (
    DEFAULT_EXPERIMENT_BASE,
    MODEL_PRESETS,
    PRESET_CHOICES,
)

print("유틸 로드 완료. 다음 셀에서 ACTIVE_PRESET / YAML을 설정하세요.")

## 0. 실험 설정 (`ACTIVE_PRESET` · YAML · 프롬프트 그룹)

- **`ACTIVE_PRESET`**: `notebook_presets.py`의 `MODEL_PRESETS` 키 중 하나.
- **`YAML_OVERRIDE`**: `configs/notebook_neuron_compare.yaml`이 있으면 그 위에 병합 (`preset` 키로 프리셋 덮어쓰기 가능).
- 프롬프트 그룹은 **2개 이상**이면 앞의 두 그룹을 비교에 사용합니다 (`P0`, `P1`).

In [ ]:
ROOT = project_root_from_notebook(Path.cwd())
YAML_OVERRIDE = ROOT / "configs" / "notebook_neuron_compare.yaml"

# === 1) 프리셋 선택 (가장 자주 바꾸는 값) ===
ACTIVE_PRESET = "qwen_reasoning_pair"

if ACTIVE_PRESET not in MODEL_PRESETS:
    raise ValueError(f"알 수 없는 프리셋 {ACTIVE_PRESET!r}. 가능: {PRESET_CHOICES}")

yaml_data: dict = {}
if YAML_OVERRIDE.is_file():
    with YAML_OVERRIDE.open(encoding="utf-8") as f:
        yaml_data = yaml.safe_load(f) or {}
    if isinstance(yaml_data.get("preset"), str):
        ACTIVE_PRESET = yaml_data["preset"]
        if ACTIVE_PRESET not in MODEL_PRESETS:
            raise ValueError(f"YAML preset {ACTIVE_PRESET!r} 가 MODEL_PRESETS 에 없습니다.")

EXP = deep_merge(deepcopy(DEFAULT_EXPERIMENT_BASE), MODEL_PRESETS[ACTIVE_PRESET])
EXP = deep_merge(EXP, {k: v for k, v in yaml_data.items() if k != "preset"})

MODELS = EXP["models"]
PROMPTS = deepcopy(EXP["prompt_groups"])
_mpg = EXP["analysis"]["max_prompts_per_group"]
if _mpg is not None:
    for _k in list(PROMPTS.keys()):
        PROMPTS[_k] = PROMPTS[_k][:_mpg]

PROMPT_KEYS = list(PROMPTS.keys())
if len(PROMPT_KEYS) < 2:
    raise ValueError("프롬프트 그룹이 2개 이상이어야 비교(P0 vs P1)가 가능합니다.")
P0, P1 = PROMPT_KEYS[0], PROMPT_KEYS[1]

DEVICE = EXP["runtime"]["device"] or ("cuda" if torch.cuda.is_available() else "cpu")
print(f"ACTIVE_PRESET = {ACTIVE_PRESET!r}   DEVICE = {DEVICE}")
print("모델:", MODELS)
print("프롬프트 그룹:", {k: len(v) for k, v in PROMPTS.items()}, "  비교:", repr(P0), "vs", repr(P1))
print("analysis:", EXP["analysis"])

## 1. 모델 로드

In [ ]:
def _get_layers(model):
    for attr in ["model.layers", "transformer.h"]:
        obj = model
        try:
            for a in attr.split("."): obj = getattr(obj, a)
            return obj
        except AttributeError:
            continue
    raise ValueError("블록을 찾을 수 없음")

DTYPE = dtype_from_str(EXP["runtime"]["dtype"])
TRC = bool(EXP["runtime"]["trust_remote_code"])

models, tokenizers = {}, {}

for tag, name in MODELS.items():
    print(f"Loading {tag}: {name} ...")
    tokenizers[tag] = AutoTokenizer.from_pretrained(name, trust_remote_code=TRC)
    models[tag] = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype=DTYPE,
        trust_remote_code=TRC,
    ).to(DEVICE).eval()

for tag, m in models.items():
    n_layers = len(_get_layers(m))
    d_model  = m.config.hidden_size
    print(f"  {tag}: layers={n_layers}, d_model={d_model}, params={sum(p.numel() for p in m.parameters())/1e6:.1f}M")

## 2. 활성화 캡처 함수

In [ ]:
def capture_activations(model, tokenizer, texts, device=DEVICE, max_length=None):
    """모든 레이어의 hidden state를 캡처.

    Returns: dict[int, Tensor(total_tokens, d_model)]
    """
    max_length = max_length if max_length is not None else int(EXP["runtime"]["max_length"])
    layers = _get_layers(model)
    n_layers = len(layers)
    storage = {}

    def hook_fn(name):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            storage[name] = h.detach().cpu()
        return hook

    handles = [layers[i].register_forward_hook(hook_fn(i)) for i in range(n_layers)]

    collected = {i: [] for i in range(n_layers)}
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
            enc = {k: v.to(device) for k, v in enc.items()}
            model(**enc)
            for i in range(n_layers):
                collected[i].append(storage[i][0])  # (seq, d_model)

    for h in handles:
        h.remove()

    return {i: torch.cat(vecs, dim=0) for i, vecs in collected.items()}

print("capture_activations 정의 완료 (max_length = EXP['runtime']['max_length'])")

## 3. 전체 조건 실행 (모델 수 × 프롬프트 그룹 수)

In [ ]:
results = {}  # key: (model_tag, prompt_tag) → dict[layer, Tensor]

for model_tag in MODELS:
    for prompt_tag, texts in PROMPTS.items():
        key = (model_tag, prompt_tag)
        print(f"▶ {model_tag} × {prompt_tag} ({len(texts)} prompts) ...")
        results[key] = capture_activations(
            models[model_tag], tokenizers[model_tag], texts
        )

sample_key = list(results.keys())[0]
n_layers = len(results[sample_key])
d_model  = results[sample_key][0].shape[-1]
TARGET_LAYER = resolve_target_layer(EXP["analysis"]["target_layer"], n_layers)
print(f"\n완료: {len(results)} 조건, {n_layers} layers, d_model={d_model}")
print(f"TARGET_LAYER = {TARGET_LAYER}   (설정: {EXP['analysis']['target_layer']!r})")

## 4. 레이어별 활성화 프로파일 (L2 norm)

In [ ]:
layer_idx = list(range(n_layers))

fig = go.Figure()
for (mtag, ptag), acts in results.items():
    l2 = [acts[i].float().norm(dim=-1).mean().item() for i in layer_idx]
    fig.add_trace(go.Scatter(
        x=layer_idx, y=l2,
        mode="lines+markers",
        name=f"{mtag} / {ptag}",
    ))

fig.update_layout(
    title="레이어별 평균 L2 Norm (잔차 스트림)",
    xaxis_title="Layer",
    yaxis_title="Mean L2 Norm",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## 5. 질문 그룹 간(P1−P0) 레이어별 L2 차이

In [ ]:
# 같은 모델에서 P1 vs P0 레이어별 L2 차이
_cols = max(1, len(MODELS))
fig = make_subplots(
    rows=1,
    cols=_cols,
    subplot_titles=list(MODELS.keys()),
)

for col, mtag in enumerate(MODELS, start=1):
    l2_p0 = [results[(mtag, P0)][i].float().norm(dim=-1).mean().item() for i in layer_idx]
    l2_p1 = [results[(mtag, P1)][i].float().norm(dim=-1).mean().item() for i in layer_idx]
    diff  = [b - a for a, b in zip(l2_p0, l2_p1)]

    fig.add_trace(
        go.Bar(x=layer_idx, y=diff, name=f"{mtag}: {P1} − {P0}"),
        row=1,
        col=col,
    )

fig.update_layout(
    title=f"질문 그룹에 따른 L2 차이 ({P1} − {P0})",
    template="plotly_white",
    height=400,
)
fig.show()

## 6. 특정 레이어의 뉴런별 활성화 히트맵

In [ ]:
TOPN = int(EXP["analysis"]["heatmap_top_neurons"])

cond_labels = [f"{m}/{p}" for m in MODELS for p in PROMPTS]
mean_acts   = []

for mtag in MODELS:
    for ptag in PROMPTS:
        h = results[(mtag, ptag)][TARGET_LAYER].float()
        mean_acts.append(h.mean(dim=0).numpy())

matrix = np.stack(mean_acts)

var_across = matrix.var(axis=0)
top_ix = np.argsort(var_across)[::-1][:TOPN]

fig = go.Figure(data=go.Heatmap(
    z=matrix[:, top_ix],
    x=[str(i) for i in top_ix],
    y=cond_labels,
    colorscale="RdBu_r",
    zmid=0,
))
fig.update_layout(
    title=f"Layer {TARGET_LAYER} — 뉴런별 평균 활성화 (조건 간 분산 상위 {TOPN})",
    xaxis_title="Neuron index",
    template="plotly_white",
    height=350,
)
fig.show()

## 7. 차등 뉴런 분석 (primary 태그 기준 P0 vs P1)

In [ ]:
TAG_PRI = EXP["tags_for_sae_plot"]["primary"]
TAG_BL = EXP["tags_for_sae_plot"]["baseline"]
KDIFF = int(EXP["analysis"]["differential_top_k"])

if TAG_PRI not in MODELS:
    raise KeyError(f"tags_for_sae_plot.primary={TAG_PRI!r} 가 MODELS 에 없습니다.")

h_pri_p0 = results[(TAG_PRI, P0)][TARGET_LAYER]
h_pri_p1 = results[(TAG_PRI, P1)][TARGET_LAYER]

diff_idx, diff_vals = differential_neurons(h_pri_p0, h_pri_p1, k=KDIFF)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[str(i) for i in diff_idx],
    y=[diff_vals[i] for i in diff_idx],
    marker_color="crimson",
))
fig.update_layout(
    title=f"[{TAG_PRI}] Layer {TARGET_LAYER} — |mean({P0}) − mean({P1})| 상위 {KDIFF} 뉴런",
    xaxis_title="Neuron index",
    yaxis_title=f"|mean({P0}) − mean({P1})|",
    template="plotly_white",
)
fig.show()

# baseline 모델이 따로 있으면 같은 뉴런 인덱스에서 방향 비교
if TAG_BL in MODELS and TAG_BL != TAG_PRI:
    h_bl_p0 = results[(TAG_BL, P0)][TARGET_LAYER]
    h_bl_p1 = results[(TAG_BL, P1)][TARGET_LAYER]

    fig2 = go.Figure()
    for label, ha, hb in [
        (TAG_PRI, h_pri_p0, h_pri_p1),
        (TAG_BL, h_bl_p0, h_bl_p1),
    ]:
        mean_a = ha.float().mean(dim=0).numpy()
        mean_b = hb.float().mean(dim=0).numpy()
        delta  = (mean_b - mean_a)[diff_idx]
        fig2.add_trace(go.Bar(
            x=[str(i) for i in diff_idx],
            y=delta.tolist(),
            name=label,
        ))

    fig2.update_layout(
        title=f"상위 {KDIFF} 차등 뉴런: ({P1} − {P0}) mean activation",
        xaxis_title="Neuron index",
        yaxis_title="Δ mean activation",
        barmode="group",
        template="plotly_white",
    )
    fig2.show()
else:
    print("baseline 이 없거나 primary 와 동일 — 그룹 막대 비교 생략")

## 8. 조건 간 코사인 유사도

In [ ]:
hiddens = {f"{m}/{p}": results[(m, p)][TARGET_LAYER] for m in MODELS for p in PROMPTS}
labels, sim = cosine_similarity_matrix(hiddens)

fig = go.Figure(data=go.Heatmap(
    z=sim, x=labels, y=labels,
    colorscale="Blues", zmin=0, zmax=1,
    text=np.round(sim, 3).astype(str), texttemplate="%{text}",
))
fig.update_layout(
    title=f"Layer {TARGET_LAYER} 조건 간 코사인 유사도",
    template="plotly_white", height=450, width=550,
)
fig.show()

## 9. (옵션) Custom SAE 학습

In [ ]:
SAE_LAYER = TARGET_LAYER
_st = EXP["analysis"]["sae_train"]

train_acts = torch.cat([
    results[(m, p)][SAE_LAYER] for m in MODELS for p in PROMPTS
], dim=0).float()

print(f"SAE 학습 데이터: {train_acts.shape}  (tokens × d_model)")

sae_cfg = SAETrainConfig(
    d_input=train_acts.shape[-1],
    d_hidden=train_acts.shape[-1] * 4,
    k=int(_st["k"]),
    lr=float(_st["lr"]),
    l1_coeff=float(_st["l1_coeff"]),
    batch_size=min(128, train_acts.shape[0]),
    epochs=int(_st["epochs"]),
    device=DEVICE,
)

trainer = SAETrainer(sae_cfg)
history = trainer.train(train_acts)
print(f"최종 loss: {history[-1]['loss']:.4f},  l0: {history[-1]['l0']:.0f}")

In [ ]:
# SAE 학습 곡선
df_hist = pd.DataFrame(history)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "L0 (활성 feature 수)"])
fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["loss"], mode="lines", name="total"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["recon"], mode="lines", name="recon"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_hist["step"], y=df_hist["l0"], mode="lines", name="l0"), row=1, col=2)
fig.update_layout(template="plotly_white", height=350, title="SAE 학습 곡선")
fig.show()

## 10. SAE Feature별 조건 간 활성화 비교

In [ ]:
sae = trainer.sae.eval()

feature_means = {}
for (mtag, ptag), acts in results.items():
    h = acts[SAE_LAYER].float().to(DEVICE)
    with torch.no_grad():
        z = sae.encode(h)
    feature_means[f"{mtag}/{ptag}"] = z.mean(dim=0).cpu().numpy()

NSF = int(EXP["analysis"]["sae_top_features"])
fm = np.stack(list(feature_means.values()))
var = fm.var(axis=0)
top_feat = np.argsort(var)[::-1][:NSF]

fig = go.Figure(data=go.Heatmap(
    z=fm[:, top_feat],
    x=[str(i) for i in top_feat],
    y=list(feature_means.keys()),
    colorscale="Viridis",
))
fig.update_layout(
    title=f"SAE Layer {SAE_LAYER} — 조건 간 분산 상위 {NSF} Feature",
    xaxis_title="SAE Feature index",
    template="plotly_white",
    height=350,
)
fig.show()

In [ ]:
z_pri_p1 = feature_means[f"{TAG_PRI}/{P1}"]
z_pri_p0 = feature_means[f"{TAG_PRI}/{P0}"]
delta_pri = z_pri_p1 - z_pri_p0

if TAG_BL in MODELS and TAG_BL != TAG_PRI:
    z_bl_p1 = feature_means[f"{TAG_BL}/{P1}"]
    z_bl_p0 = feature_means[f"{TAG_BL}/{P0}"]
    delta_bl = z_bl_p1 - z_bl_p0
    specificity = np.abs(delta_pri) - np.abs(delta_bl)
    top_specific = np.argsort(specificity)[::-1][:20]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_pri[top_specific], name=f"{TAG_PRI} Δ"))
    fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_bl[top_specific], name=f"{TAG_BL} Δ"))
    fig.update_layout(
        title=f"[{TAG_PRI} vs {TAG_BL}] {P1}−{P0} 변화가 primary 에서 더 큰 SAE feature 20개",
        xaxis_title="SAE Feature",
        yaxis_title=f"mean z({P1}) − z({P0})",
        barmode="group",
        template="plotly_white",
    )
    fig.show()
else:
    top_specific = np.argsort(np.abs(delta_pri))[::-1][:20]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=[str(i) for i in top_specific], y=delta_pri[top_specific], name=f"{TAG_PRI}"))
    fig.update_layout(
        title=f"[{TAG_PRI}] 단일 모델 — {P1}−{P0} 변화 상위 20 SAE feature",
        template="plotly_white",
    )
    fig.show()

print("\n다음 단계: activation patching 으로 인과 검증")